# King Domino Projekt - Overblik

### Arkitekturoversigt
Projektets formål er at digitalisere og point-score fysiske King Domino-spilleplader. Projektet består af fem hovedfaser:
1. **Gitteropdeling (Grid Splitting):** 
    - Opdeling af hele pladebilleder i et 5x5 logisk gitter af terræn-tiles.
2. **Feature-ekstraktion:** 
    - Behandling af tærren-tiles for at udtrække features til en 20-værdi vektor (HSV-histogram logik)
    - Gemme 20-værdi vektorerne i en csv-fil til modeltræning/analyse.
3. **Tile-multiklassificering:** 
    - Identificering af tile-terræntypen (Water, Field, Grass, Forest, Mine, Swamp, Home, Empty). 
    - Altså en multi-class terrænklassificering udført med enten SVM eller kNN
4. **Krone-detektion:** 
    - Tælle antallet af score-multiplikatorer (kroner) til stede på hver klassificeret terræn-tile ved at bruge template matching (fx med `high_res_crown.png` / `low_res_crown.png`). 
    - Behandle gråtone/blobs
    - Håndtering af gennemsigtige "Don't care"-pixels.
5. **Spillogik og Scoreberegning:** 
    - Evaluering af 2D-arrays, der repræsenterer spillepladernes 5x5-gitter.
    - Hver spilleplade (fx med Breadth-First Search algoritme) til at beregne den endelige score for regioner (forbundne terræn-tiles).
    - Baseret på Kingdomino-reglerne(fx 6 forbundne vand-tiles * 2 kroner = 12 point).
6. **Ground Truths(GT):**
    - GT_BoardPointScore
    - GT_BoardTileMatrix
    - GT_CrownsPerTile
    - GT_CrownsPerBoard


### Nuværende status

| Modul | Status | Beskrivelse |
| :--- | :---: | :--- |
| `board_split.py` | 🟢 **Færdig** | Deler korrekt hovedbilledet op i et 5x5 tile-gitter. |
| `feature_extraction.py` | 🟢 **Færdig** | Ekstraherer HSV-histogrammer fra tiles og bygger et dataset (`features.csv`). |
| `extract_tile_feature_vector.py` | 🟡 **Igangværende** | Indlæser billeder, opdeler tiles og orkestrerer feature-ekstraktion med succes, men mangler den endelige inferens- og scorings-pipeline. |
| `classify_tile.py` | 🟢 **Færdig** | Implementerer `TileClassifier` ved hjælp af en Scikit-learn k-Nearest Neighbors (kNN) model trænet på `features.csv`. |
| `detect_crown.py` | 🔴 **TODO** | Stub (`CrownDetector`). Kræver kontur/form-detektion for at tælle 0-3 gyldne kroner per tile. |
| `scoring.py` | 🔴 **TODO** | Stub (`ScoreCalculator`). Kræver BFS-algoritmen til at gange størrelsen af forbundne regioner med antal kroner. |

### Manglende dele
1. **Krone-tællelogik:** Identificering af specifikke gule konturer/former på ikke-tomme tiles for at fungere som multiplikatorer (`detect_crown.py`).
2. **Spilleregel-motor:** Implementering af BFS-gittergennemløb for at gruppere tilstødende matchende terræner og beregne den endelige score (`scoring.py`).
3. **End-to-End Integration:** Modificering af `extract_tile_feature_vector.py` til at bruge klassificeringsmodellerne og sende det resulterende 5x5 gitter af objekter til scoringsmotoren.

### Anbefalet næste skridt
**Implementer krone-detektion (`detect_crown.py`)**
Nu hvor vi kan ekstrahere features og klassificere terræntypen for hver tile med en ML-model, er den næste komponent at bygge krone-detektionen. Dette vil identificere score-multiplikatorer.

**Næste kode der skal genereres:**
Vi bør udfylde logikken indeni `CrownDetector` (`detect_crown.py`) for at detektere og tælle forekomster af gyldne kroner via template matching eller konturdetektion, med højde for potentiel støj / overlappende elementer.

## 1. Dataforbehandling og split-strategi
For at undgå datalækage laves split konsekvent på spilleplade-niveau og ikke på tile-niveau. Parrede billeder (med og uden 3D-objekter) placeres altid i samme split.

* **Samlet antal spilleplader:** 36 unikke spilleplader.
* **Ugyldige data(støj):** Billede 71, 73 og 74 er udeladt, fordi de er dubletter af billede 68, 69 og 70.
* **Dataekstraktion:** ca. 1775 tiles (71 gyldige billeder × 25 felter).

### Endelig split:
* **Hold-out testsæt:** 6 spilleplade-grupper (12 billeder i alt) reserveres til den endelige evaluering for at teste præstation af ML-model.

Test-sæt:
Spilleplader
(1,5), (19,23), (25,29), (35,39), (49,53), (67,70)

* **Træning og validering (grouped CV):** De resterende 30 spilleplade-grupper bruges til 5-fold grouped cross-validation.
  * Hver fold indeholder 6 spilleplade-grupper.
  * I hver CV-runde bruges 4 folds til træning og 1 fold til validering.

Fold 1:
Spilleplader 
(4,8), (20,24), (34,38), (42,46), (48,52), (65,72)

Fold 2:
Spilleplader
(2,6), (18,22), (28,32), (36,40), (51,55), (58,62)

Fold 3:
Spilleplader
(10,14), (11,15), (26,30), (41,44), (57,61), (64,68)

Fold 4:
Spilleplader
(3,7), (17,21), (27,31), (43,47), (50,54), (59,63)

Fold 5:
Spilleplader
(9,13), (12,16), (33,37), (45), (56,60), (66,69)
